# Day 3 - Code breaks, block 3: evaluation - does it really work?

Short code breaks that go with the slides (3-5 minutes each):
1. **run** the cell and look at the output,
2. **change** the number marked with `# 👈`, run again,
3. open **👀 What you should see** to check.

A **CPU runtime is enough**. You cannot break anything - if something looks strange: *Runtime → Restart session*,
then run the first two cells again.

**Data:** the exercise numbers from the slides, the *Iris* flowers and made-up ("synthetic") data - no EEG download.

## Setup - run these two cells first
The first cell loads PyTorch. PyTorch has no built-in metrics, so we use scikit-learn **only for the metrics**
(accuracy, sensitivity, AUROC, ...).

In [ ]:
import torch
import torch.nn as nn
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score)

print("ready - PyTorch", torch.__version__)

The second cell defines small helpers. **Just run it - you do not need to read it.**
- `train_classifier(...)` - trains a small PyTorch network (or, with `hidden=0`, a single `nn.Linear` layer)
- `accuracy(...)` - share of correct predictions
- `show_metrics(...)` - prints the confusion matrix numbers and all metrics from the slides
- `load_iris_split(...)` - Iris flowers as tensors, split into training and test
- `imbalanced_scores(...)`, `show_threshold(...)`, `show_interval(...)` - for code breaks 2 and 4

In [ ]:
from sklearn.datasets import load_iris, make_classification

torch.set_num_threads(1)  # tiny networks: one thread is faster


def train_classifier(X, y, hidden=16, n_out=2, epochs=300, lr=0.01, seed=0):
    torch.manual_seed(seed)
    if hidden == 0:
        model = nn.Linear(X.shape[1], n_out)
    else:
        model = nn.Sequential(nn.Linear(X.shape[1], hidden), nn.ReLU(), nn.Linear(hidden, n_out))
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):                                   # full batch: all windows at once
        loss = loss_fn(model(X), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return model


def accuracy(model, X, y):
    with torch.no_grad():
        return (model(X).argmax(1) == y).float().mean().item()


def show_metrics(name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    print("model", name, ":  TP", tp, " FN", fn, " FP", fp, " TN", tn)
    print("   accuracy         ", round(100 * accuracy_score(y_true, y_pred)), "%")
    print("   sensitivity      ", round(100 * recall_score(y_true, y_pred, zero_division=0)), "%")
    print("   specificity      ", round(100 * recall_score(y_true, y_pred, pos_label=0, zero_division=0)), "%")
    print("   precision        ", round(100 * precision_score(y_true, y_pred, zero_division=0)), "%")
    print("   F1               ", round(f1_score(y_true, y_pred, zero_division=0), 2))
    print("   balanced accuracy", round(100 * balanced_accuracy_score(y_true, y_pred)), "%")


def load_iris_split(n_measurements=4, seed=3):
    iris = load_iris()
    X = torch.tensor(iris.data[:, :n_measurements], dtype=torch.float32)
    y = torch.tensor(iris.target)
    g = torch.Generator().manual_seed(seed)
    test = torch.zeros(150, dtype=torch.bool)
    for c in range(3):                                            # 15 test flowers per class
        idx = torch.nonzero(y == c).flatten()
        test[idx[torch.randperm(50, generator=g)[:15]]] = True
    mean, std = X[~test].mean(0), X[~test].std(0)                 # standardise with the training part only
    X = (X - mean) / std
    return X[~test], X[test], y[~test], y[test]


def imbalanced_scores(class_sep=1.0, n_test=2000):
    X, y = make_classification(n_samples=4000, n_features=10, weights=[0.98], class_sep=class_sep,
                               flip_y=0, random_state=0)
    X, y = torch.tensor(X, dtype=torch.float32), torch.tensor(y)
    model = train_classifier(X[::2], y[::2], hidden=0)          # every 2nd window for training, a linear model
    with torch.no_grad():
        scores = model(X[1::2]).softmax(1)[:, 1]                 # score = probability of "positive"
    return y[1::2][:n_test], scores[:n_test]


def show_threshold(y_true, scores, threshold):
    tn, fp, fn, tp = confusion_matrix(y_true, (scores >= threshold).long(), labels=[0, 1]).ravel()
    print("threshold", threshold, ":  sensitivity", round(100 * tp / (tp + fn)), "%",
          "  specificity", round(100 * tn / (tn + fp), 1), "%", "  false alarms", fp)


def show_interval(low, high):
    if low is None:
        print("95 % confidence interval: ? - fill in the missing line")
    else:
        print("95 % confidence interval:", round(float(low), 2), "to", round(float(high), 2))


print("helpers ready")

## 💻 Code break 1 - The confusion-matrix exercise: check your results
The exercise from the slides: 100 windows, 10 seizure (`1`), 90 background (`0`). Compare with your hand
calculation.

In [ ]:
y_true = torch.tensor([1] * 10 + [0] * 90)                        # 10 seizure, 90 background

y_pred_A = torch.tensor([1] * 8 + [0] * 2 + [1] * 9 + [0] * 81)   # TP 8, FN 2, FP 9, TN 81
y_pred_B = torch.zeros(100, dtype=torch.long)                     # always "background"
y_pred_C = torch.ones(100, dtype=torch.long)                      # always "seizure"

show_metrics("A", y_true, y_pred_A)
show_metrics("B (always background)", y_true, y_pred_B)
show_metrics("C (always seizure)", y_true, y_pred_C)

<details><summary>👀 What you should see</summary>

Exactly the table on the solution slide:
- A: accuracy **89 %**, sensitivity **80 %**, specificity **90 %**, precision **47 %**, F1 **0.59**, balanced **85 %**
- B: **90 %**, **0 %**, **100 %**, 0 % (really 0 / 0), **0.0**, **50 %**
- C: **10 %**, **100 %**, **0 %**, **10 %**, **0.18**, **50 %**
</details>

Now a small PyTorch network on the **Iris** flowers. 3 classes → the confusion matrix is 3 × 3.

In [ ]:
# import the modules
from sklearn.metrics import confusion_matrix, classification_report

N_MEASUREMENTS = 4   # 👈 how many of the 4 flower measurements the model gets

X_tr, X_te, y_tr, y_te = load_iris_split(N_MEASUREMENTS)
model = train_classifier(X_tr, y_tr, n_out=3)
with torch.no_grad():
    y_hat = model(X_te).argmax(1)       # predicted flower

print(confusion_matrix(y_te, y_hat))   # rows = true flower, columns = predicted flower
print(classification_report(y_te, y_hat, target_names=["setosa", "versicolor", "virginica"]))

🔧 **Change it:**
1. Which flowers does the model mix up? (Numbers outside the diagonal.)
2. `N_MEASUREMENTS = 2` - only the two sepal measurements. What happens?

<details><summary>👀 What you should see</summary>

- accuracy **96 %**: setosa and virginica always right, 2 versicolor called virginica
- 2 measurements: accuracy **80 %** (9 mix-ups) - versicolor and virginica look much more alike
</details>

## 💻 Code break 2 - Choosing the threshold, AUROC vs. AUPRC
Made-up, very imbalanced data (like seizures): 2000 test windows, only 2 % positive. A single `nn.Linear` layer
gives a **score** between 0 and 1 - the threshold turns it into yes / no.

In [ ]:
CLASS_SEP = 1.0    # 👈 how easy the two classes are to tell apart

y_test, scores = imbalanced_scores(CLASS_SEP)
print(int(y_test.sum()), "positives of", len(y_test), "test windows")

for threshold in [0.1, 0.3, 0.5, 0.7, 0.9]:
    show_threshold(y_test, scores, threshold)

print("AUROC:", round(roc_auc_score(y_test, scores), 2), "  (guessing = 0.5)")
print("AUPRC:", round(average_precision_score(y_test, scores), 2), "  (guessing = 0.02)")

🔧 **Change it:**
1. A missed seizure is much worse than a false alarm. Which threshold do you choose?
2. `CLASS_SEP = 2.0` - an easier problem. How do AUROC and AUPRC change?

<details><summary>👀 What you should see</summary>

- 36 positives. Threshold 0.1: sensitivity **92 %**, but **218** false alarms; 0.5: **22 %**, **11** false alarms;
  0.9: **6 %**, **0** false alarms → for "do not miss seizures" take a low threshold
- specificity stays at **89 %** or more at every threshold - it hides the false alarms
- **AUROC 0.96** looks great, **AUPRC 0.34** is much more honest
- `CLASS_SEP = 2.0`: AUROC **1.0**, AUPRC **0.98**
</details>

## 💻 Code break 3 - One run is one sample: several seeds
The same small network on the same Iris split. Only the **seed** (the random start of the weights) changes.

In [ ]:
EPOCHS = 20        # 👈 training steps
N_SEEDS = 5        # 👈 how many seeds

X_tr, X_te, y_tr, y_te = load_iris_split(seed=1)
accs = []
for seed in range(N_SEEDS):
    net = train_classifier(X_tr, y_tr, hidden=8, n_out=3, epochs=EPOCHS, seed=seed)
    accs.append(accuracy(net, X_te, y_te))
    print("seed", seed, ": test accuracy", round(100 * accs[-1]), "%")

accs = torch.tensor(accs)
print("mean:", round(100 * accs.mean().item()), "%   std:", round(100 * accs.std().item()), "%")

🔧 **Change it:**
1. `EPOCHS = 300` - train longer. Do the seeds agree more?
2. `N_SEEDS = 10`

<details><summary>👀 What you should see</summary>

- 5 seeds: between **71 %** and **87 %** (mean **81 %**, std **6 %**) - only the random start differs!
  Reporting only the best seed would give 87 %.
- `EPOCHS = 300`: all seeds reach **100 %** - a well-trained network is much more stable
- 10 seeds: **71 %** to **87 %**, mean **81 %**, std **5 %**
</details>

## 💻 Code break 4 - How sure are we? A bootstrap confidence interval
The data from code break 2. We draw the test windows 2000 times **with replacement** and compute the AUPRC each
time. The middle 95 % of the results = the 95 % confidence interval.

✏️ **Fill in one line:** replace `pass` by `low, high = ...` - use `torch.quantile(results, torch.tensor([0.025, 0.975]))`,
it gives the 2.5 % point and the 97.5 % point of all results.

In [ ]:
N_TEST = 2000      # 👈 size of the test set

y_test, scores = imbalanced_scores(n_test=N_TEST)
torch.manual_seed(0)
results = []
for i in range(2000):
    idx = torch.randint(0, len(y_test), (len(y_test),))   # draw test windows with replacement
    results.append(average_precision_score(y_test[idx], scores[idx]))
results = torch.tensor(results)

low, high = None, None
pass  # YOUR CODE HERE
print("AUPRC:", round(average_precision_score(y_test, scores), 2))
show_interval(low, high)

🔧 **Change it:**
1. `N_TEST = 500` - a smaller test set. Wider or narrower interval?

<details><summary>👀 What you should see</summary>

- AUPRC **0.34**, interval **0.22 to 0.49** - wide, because there are only 36 positive windows
- 500 test windows (only 8 positives): AUPRC **0.29**, interval **0.11 to 0.62** - says almost nothing
- real EEG windows are not independent: with real data, draw recordings or patients instead of windows
</details>